In [4]:
# adam_burgers_periodic_stable.py
# ------------------------------------------------------------
# Pure-JAX Adam baseline for 1D Burgers (from burgers.mat)
# PDE:  u_t + u u_x - nu u_xx = 0
# BC:   periodic: u(x_min,t)=u(x_max,t), u_x(x_min,t)=u_x(x_max,t)
# IC:   u(x,t_min)=u0(x) from burgers.mat (interpolated)
#
# Option A (resample every iteration):
#   PDE: 2000 interior points
#   BC:  200 time samples (paired L/R), includes value+derivative
#   IC:  200 x samples at t=t_min
#
# Improvements vs your earlier version:
#   - FIX JAX IndexError: predict_u is NOT jitted (or shapes static)
#   - Smaller LR: default lr=1e-4
#   - Global grad-norm clipping (default clip=1.0)
#   - Match SQP PDE weight: w_pde=10.0 (edit if you want)
#
# Saves:
#   burgers_adam_theta.npy
#   hist_adam_*.npy
# ------------------------------------------------------------

import os
import math
import time
import numpy as np
import scipy.io

import jax
import jax.numpy as jnp
from jax import random, grad, vmap, hessian

# If you want to choose GPU, set BEFORE importing jax in a real script.
# Keeping here because you had it; it may be ignored if jax already imported.
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0")

# -----------------------------
# Precision / dtype
# -----------------------------
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32

# -----------------------------
# Batch sizes
# -----------------------------
B_PDE = 9000
B_BC  = 2000
B_IC  = 1000

# -----------------------------
# Loss weights (match your SQP w_pde_obj)
# -----------------------------
W_PDE = 1   # set to 1.0 if you want unweighted

# -----------------------------
# Data loading (burgers.mat)
# -----------------------------
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)
    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]  # (nt, nx)
    nu = float(np.array(d["nu"]).squeeze())
    return t, x, usol, nu

def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape

# -----------------------------
# MLP
# -----------------------------
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def mlp_apply(params, x):
    h = x
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)
    return h  # (N,1)

# -----------------------------
# Flatten/unflatten (theta vector)
# -----------------------------
def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    theta = jnp.concatenate(flat_parts).astype(DTYPE)
    return theta, tuple(shapes)

def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx: idx + W_size].reshape(W_shape); idx += W_size
        b = theta[idx: idx + b_size].reshape(b_shape); idx += b_size
        params.append({"W": W, "b": b})
    return params

# -----------------------------
# Burgers residual and helpers
# -----------------------------
def pde_residual_unscaled(params, X_f, nu):
    # X_f: (N,2) columns [x,t]
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u  = vmap(u_fun)(X_f)
    du = vmap(grad(u_fun))(X_f)
    H  = vmap(hessian(u_fun))(X_f)

    u_x  = du[:, 0]
    u_t  = du[:, 1]
    u_xx = H[:, 0, 0]
    return u_t + u * u_x - DTYPE(nu) * u_xx

@jax.jit
def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]
    u  = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux

# -----------------------------
# Sampling (resample each iter)
# -----------------------------
def _sample_uniform(key, shape, lo, hi):
    return random.uniform(key, shape, minval=DTYPE(lo), maxval=DTYPE(hi), dtype=DTYPE)

def sample_pde_uniform(key, x_min, x_max, t_min, t_max, B=B_PDE):
    kx, kt = random.split(key, 2)
    x = _sample_uniform(kx, (B, 1), x_min, x_max)
    t = _sample_uniform(kt, (B, 1), t_min, t_max)
    return jnp.concatenate([x, t], axis=1)  # (B,2)

def sample_bc_periodic(key, x_min, x_max, t_min, t_max, B=B_BC):
    t = _sample_uniform(key, (B, 1), t_min, t_max)
    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(t), t], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(t), t], axis=1)
    return XL, XR

def sample_ic(key, x_min, x_max, t0, x_grid, u0_grid, B=B_IC):
    x = _sample_uniform(key, (B, 1), x_min, x_max)
    t = DTYPE(t0) * jnp.ones_like(x)
    Xic = jnp.concatenate([x, t], axis=1)
    u0 = jnp.interp(x[:, 0], x_grid, u0_grid)
    return Xic, u0

# -----------------------------
# Adam loss (weights: W_PDE for PDE, others 1)
# -----------------------------
def adam_loss_parts(theta, shapes, key,
                    x_min, x_max, t_min, t_max,
                    x_grid, u0_grid, nu):
    """
    Returns:
      total, (Lpde, Lbc_val, Lbc_der, Lic)
    total = W_PDE*Lpde + Lbc_val + Lbc_der + Lic
    """
    params = unflatten_params(theta, shapes)
    key, k_pde, k_bc, k_ic = random.split(key, 4)

    # PDE
    Xpde  = sample_pde_uniform(k_pde, x_min, x_max, t_min, t_max, B=B_PDE)
    r_pde = pde_residual_unscaled(params, Xpde, nu)
    Lpde  = jnp.mean(r_pde**2)

    # periodic BC
    XL, XR = sample_bc_periodic(k_bc, x_min, x_max, t_min, t_max, B=B_BC)
    uL, uxL = u_and_ux(params, XL)
    uR, uxR = u_and_ux(params, XR)
    Lbc_val = jnp.mean((uL - uR)**2)
    Lbc_der = jnp.mean((uxL - uxR)**2)

    # IC at t=t_min
    Xic, u0 = sample_ic(k_ic, x_min, x_max, t_min, x_grid, u0_grid, B=B_IC)
    u_ic = mlp_apply(params, Xic)[:, 0]
    Lic = jnp.mean((u_ic - u0)**2)

    total = DTYPE(W_PDE) * Lpde + Lbc_val + Lbc_der + Lic
    return total, (Lpde, Lbc_val, Lbc_der, Lic)

# -----------------------------
# Pure JAX Adam update (theta vector)
# -----------------------------
@jax.jit
def adam_update(theta, g, m, v, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1.0 - b1) * g
    v = b2 * v + (1.0 - b2) * (g * g)
    t = t + 1
    mhat = m / (1.0 - b1**t)
    vhat = v / (1.0 - b2**t)
    theta = theta - lr * mhat / (jnp.sqrt(vhat) + eps)
    return theta, m, v, t

# -----------------------------
# Training with grad clipping
# -----------------------------
def train_adam(theta0, shapes,
               x_min, x_max, t_min, t_max,
               x_grid, u0_grid, nu,
               seed=0,
               n_steps=10_000,
               lr=1e-4,
               grad_clip=1.0,
               b1=0.9, b2=0.999, eps=1e-8,
               print_every=200):

    key = random.PRNGKey(seed)
    theta = theta0
    m = jnp.zeros_like(theta)
    v = jnp.zeros_like(theta)
    t = jnp.array(0, dtype=jnp.int32)

    hist = {"total": [], "pde": [], "bc_val": [], "bc_der": [], "ic": [], "gnorm": []}

    @jax.jit
    def step(theta, m, v, t, key):
        (loss_val, parts), g = jax.value_and_grad(
            lambda th: adam_loss_parts(th, shapes, key,
                                      x_min, x_max, t_min, t_max,
                                      x_grid, u0_grid, nu),
            has_aux=True
        )(theta)

        # global-norm clip
        gnorm = jnp.linalg.norm(g)
        scale = jnp.minimum(DTYPE(1.0), DTYPE(grad_clip) / (gnorm + DTYPE(1e-12)))
        g = g * scale

        theta, m, v, t = adam_update(theta, g, m, v, t, lr, b1=b1, b2=b2, eps=eps)
        return theta, m, v, t, loss_val, parts, gnorm

    t0_wall = time.time()
    for k in range(1, n_steps + 1):
        key, kstep = random.split(key, 2)
        theta, m, v, t, total, (Lpde, Lbcv, Lbcd, Lic), gnorm = step(theta, m, v, t, kstep)

        hist["total"].append(float(total))
        hist["pde"].append(float(Lpde))
        hist["bc_val"].append(float(Lbcv))
        hist["bc_der"].append(float(Lbcd))
        hist["ic"].append(float(Lic))
        hist["gnorm"].append(float(gnorm))

        if k % print_every == 0:
            print(f"[adam k={k:6d}] total={float(total):.3e} "
                  f"pde={float(Lpde):.3e} bcv={float(Lbcv):.3e} "
                  f"bcd={float(Lbcd):.3e} ic={float(Lic):.3e} "
                  f"| gnorm={float(gnorm):.2e}")

    print(f"[adam done] elapsed={time.time() - t0_wall:.2f}s")
    return theta, hist

# -----------------------------
# Eval on full grid (NO JIT to avoid dynamic slicing issue)
# -----------------------------
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]

def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.array(X_grid_np, dtype=DTYPE)
    u_pred = np.array(predict_u(theta, shapes, X_grid)).reshape(grid_shape)
    u_true = usol_np
    mse = float(np.mean((u_pred - u_true) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12))
    return mse, rel_l2, u_pred

def plot_heatmaps(x_np, t_np, u_true, u_pred, title_prefix="ADAM"):
    import matplotlib.pyplot as plt
    err = u_pred - u_true
    plt.figure(); plt.pcolormesh(x_np, t_np, u_true, shading="auto"); plt.colorbar()
    plt.title(f"{title_prefix}: True"); plt.xlabel("x"); plt.ylabel("t"); plt.show()
    plt.figure(); plt.pcolormesh(x_np, t_np, u_pred, shading="auto"); plt.colorbar()
    plt.title(f"{title_prefix}: Pred"); plt.xlabel("x"); plt.ylabel("t"); plt.show()
    plt.figure(); plt.pcolormesh(x_np, t_np, np.abs(err), shading="auto"); plt.colorbar()
    plt.title(f"{title_prefix}: |Error|"); plt.xlabel("x"); plt.ylabel("t"); plt.show()

# -----------------------------
# main
# -----------------------------
def main():
    mat_path = "data/burgers.mat"
    t_np, x_np, usol_np, nu = load_burgers_mat(mat_path)
    print("loaded:", mat_path, "t", t_np.shape, "x", x_np.shape, "usol", usol_np.shape, "nu", nu)

    x_min, x_max = float(x_np.min()), float(x_np.max())
    t_min, t_max = float(t_np.min()), float(t_np.max())
    print(f"Domain: x in [{x_min},{x_max}], t in [{t_min},{t_max}]")

    # IC grid
    x_grid = jnp.array(x_np, dtype=DTYPE)
    u0_grid = jnp.array(usol_np[0, :], dtype=DTYPE)

    # network
    hidden_dim = 50
    num_hidden = 4
    layer_sizes = [2] + [hidden_dim] * num_hidden + [1]

    key = random.PRNGKey(0)
    params0 = init_mlp_params(key, layer_sizes)
    theta0, shapes = flatten_params(params0)

    # train Adam (stable defaults)
    theta_adam, hist = train_adam(
        theta0, shapes,
        x_min, x_max, t_min, t_max,
        x_grid, u0_grid, nu,
        seed=0,
        n_steps=50000,
        lr=1e-4,
        grad_clip=2.0,
        print_every=1
    )

    # save
    np.save("burgers_adam_theta.npy", np.array(theta_adam))
    np.save("hist_adam_total.npy", np.array(hist["total"]))
    np.save("hist_adam_pde.npy",   np.array(hist["pde"]))
    np.save("hist_adam_bcv.npy",   np.array(hist["bc_val"]))
    np.save("hist_adam_bcd.npy",   np.array(hist["bc_der"]))
    np.save("hist_adam_ic.npy",    np.array(hist["ic"]))
    np.save("hist_adam_gnorm.npy", np.array(hist["gnorm"]))
    print("saved: burgers_adam_theta.npy and hist_adam_*.npy")

    # eval
    mse, rel_l2, u_pred = eval_full_grid(theta_adam, shapes, x_np, t_np, usol_np)
    print(f"[ADAM EVAL] full-grid MSE={mse:.3e}, relL2={rel_l2:.3e}")

    if os.environ.get("PLOT", "0") == "1":
        plot_heatmaps(x_np, t_np, usol_np, u_pred, title_prefix="ADAM")


if __name__ == "__main__":
    main()

loaded: data/burgers.mat t (201,) x (512,) usol (201, 512) nu 0.003183098861837907
Domain: x in [-1.0,1.0], t in [0.0,1.0]
[adam k=     1] total=4.525e+00 pde=8.442e-01 bcv=1.170e+00 bcd=6.148e-01 ic=1.896e+00 | gnorm=6.23e+01
[adam k=     2] total=4.130e+00 pde=7.786e-01 bcv=1.016e+00 bcd=5.579e-01 ic=1.778e+00 | gnorm=5.86e+01
[adam k=     3] total=3.830e+00 pde=7.353e-01 bcv=8.736e-01 bcd=5.218e-01 ic=1.700e+00 | gnorm=5.56e+01
[adam k=     4] total=3.513e+00 pde=6.890e-01 bcv=7.460e-01 bcd=5.075e-01 ic=1.570e+00 | gnorm=5.22e+01
[adam k=     5] total=3.241e+00 pde=6.332e-01 bcv=6.286e-01 bcd=4.754e-01 ic=1.504e+00 | gnorm=4.94e+01
[adam k=     6] total=2.961e+00 pde=5.437e-01 bcv=5.130e-01 bcd=4.080e-01 ic=1.496e+00 | gnorm=4.54e+01
[adam k=     7] total=2.672e+00 pde=5.041e-01 bcv=4.237e-01 bcd=4.112e-01 ic=1.333e+00 | gnorm=4.20e+01
[adam k=     8] total=2.532e+00 pde=4.951e-01 bcv=3.389e-01 bcd=3.834e-01 ic=1.314e+00 | gnorm=3.96e+01
[adam k=     9] total=2.236e+00 pde=4.312e-01

In [0]:
relL2=5.576e-02

In [0]:
pkill -9 -u $(whoami) python